In [1]:
import pandas as pd

In [4]:
price_mapped_nutrients_df = pd.read_csv("./input_stage2/price_mapped_nutrients.csv")
price_mapped_nutrients_df

,fdc_id,description,product,item_prices,unpriced_items,priced_items,total_items,priced_share,adj,verb,total_price,protein_g,fat_g,carb_g,energy_kcal,fiber_g,sodium_mg,cholesterol_mg,satfat_g,sugars_g
0,167512,"Pillsbury Golden Layer Buttermilk Biscuits, Ar...","['layer buttermilk biscuits', 'dough']",{'layer buttermilk biscuits': 4.5},['dough'],1,2,0.5,"['Golden', 'Artificial']",['refrigerated'],0.00,5.88,13.2400,41.1800,307.0,1.2,1059.0,0.00,2.9410,5.88
1,167516,"Waffles, buttermilk, frozen, ready-to-heat",['waffles'],{'waffles': 6.85},[],1,1,1.0,['ready'],['frozen'],6.85,6.58,9.2200,41.0500,273.0,2.2,621.0,15.00,1.8980,4.30
2,167517,"Waffle, buttermilk, frozen, ready-to-heat, toa...",['waffle'],{'waffle': 6.85},[],1,1,1.0,['ready'],"['frozen', 'toasted']",6.85,7.42,9.4900,48.3900,309.0,2.6,710.0,13.00,2.2750,4.41
3,167518,"Waffle, buttermilk, frozen, ready-to-heat, mic...",['waffle'],{'waffle': 6.85},[],1,1,1.0,['ready'],"['frozen', 'microwaved']",6.85,6.92,9.4000,44.1600,289.0,2.4,663.0,16.00,2.0570,4.50
4,167519,"Waffle, plain, frozen, ready-to-heat, microwave",['waffle'],{'waffle': 6.85},[],1,1,1.0,"['plain', 'ready']",['frozen'],6.85,6.71,9.9100,45.4100,298.0,2.4,682.0,16.00,1.5800,5.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10840,2710811,Vegetables as ingredient in soups,"['vegetables', 'soups']","{'vegetables': 3.52, 'soups': 4.59}",[],2,2,1.0,[],[],8.11,1.87,0.3300,10.5600,50.0,2.0,15.0,0.00,0.0500,2.34
10841,2710812,Vegetables as ingredient in stews,"['vegetables', 'stews']",{'vegetables': 3.52},['stews'],1,2,0.5,[],[],0.00,1.77,0.3000,12.6700,59.0,2.6,34.0,0.00,0.0400,3.16
10842,2710813,Sauce as ingredient in hamburgers,['sauce'],{'sauce': 3.95},[],1,1,1.0,[],[],3.95,1.34,22.8500,17.1400,272.0,0.6,845.0,13.00,3.5440,13.08
10843,2710814,Industrial oil as ingredient in food,['oil'],{'oil': 8.85},[],1,1,1.0,['Industrial'],[],8.85,0.00,100.0000,0.0000,892.0,0.0,0.0,0.00,32.6720,0.00


## Multi-label recipe classes (labels.md)

Silver labels built from the description, adj/verb tokens and nutrients. Output: `input_stage2/recipe_classes.csv` — source `product`/`adj`/`verb` + nutrient columns, one 0/1 column per label, and an exclusive `generic` column for rows with no other class; plus `heat_level`, `cuisine_region`, pipe-joined `labels`.

In [ ]:
import re
import numpy as np

# Silver labels for the categories in labels.md. The rows are USDA FDC food items
# (no ingredient lists / instructions), so labels are inferred from the description,
# the adj/verb tokens and the nutrient columns. Strict dietary labels are left to rule-based filters.

def kw(*words):
    return re.compile(r"\b(?:" + "|".join(words) + r")", re.I)

text = (
    price_mapped_nutrients_df["description"].fillna("") + " | "
    + price_mapped_nutrients_df["product"].fillna("") + " | "
    + price_mapped_nutrients_df["adj"].fillna("") + " | "
    + price_mapped_nutrients_df["verb"].fillna("")
).str.lower()
desc = price_mapped_nutrients_df["description"].fillna("").str.lower()
n = price_mapped_nutrients_df.fillna({c: 0 for c in
    ["protein_g", "fat_g", "carb_g", "energy_kcal", "fiber_g", "sodium_mg",
     "cholesterol_mg", "satfat_g", "sugars_g"]})

def has(pattern, s=text):
    return s.str.contains(pattern, regex=True)

# ---------- item-type context ----------
INFANT = has(kw("infant formula", "babyfood", "baby food", "toddler formula"))
BEVERAGE = has(kw("beverage", "drink", "juice", "soda", "soft drink", "coffee", "tea\\b", "water\\b",
                  "smoothie", "shake", "cocoa mix", "lemonade", "milk, ", "wine", "beer", "liquor", "cocktail",
                  "vodka", "whiskey", "rum\\b", "gin\\b", "punch"), desc)
ALCOHOL = has(kw("alcoholic", "wine", "beer", "ale\\b", "lager", "liquor", "liqueur", "vodka", "whiskey", "whisky",
                 "rum\\b", "gin\\b", "brandy", "tequila", "sake", "mirin", "cocktail", "margarita", "daiquiri",
                 "sangria", "eggnog.*alcohol", "bourbon", "sherry", "cooking wine", "marsala", "vanilla extract")) \
    & ~has(kw("ginger ale", "root beer", "non-?alcoholic", "birch beer", "wine vinegar"))
DESSERT = has(kw("cake", "cookie", "pie\\b", "pies\\b", "pudding", "ice cream", "candy", "candies", "brownie",
                 "frosting", "sorbet", "sherbet", "doughnut", "donut", "pastry", "pastries", "cheesecake",
                 "custard", "gelatin dessert", "fudge", "chocolate", "cupcake", "cobbler", "tart\\b", "danish",
                 "sweet roll", "eclair", "cream puff", "popsicle", "frozen novelt", "marshmallow", "toffee",
                 "caramel", "baklava", "flan\\b", "tiramisu", "mousse"))
MAIN_DISH = (has(kw("with", "entree", "meal", "dinner", "sandwich", "burger", "stew", "soup", "chili", "casserole",
                    "pizza", "burrito", "taco", "enchilada", "lasagna", "spaghetti", "macaroni", "noodles",
                    "fried rice", "stir.?fr", "curry", "pot pie", "pot roast", "meatloaf", "meatballs",
                    "fajita", "quesadilla", "tamale", "pad thai", "lo mein", "chow mein", "jambalaya",
                    "gumbo", "paella", "risotto", "goulash", "ravioli", "tortellini", "fettuccine",
                    "alfredo", "chicken or turkey", "beef and", "pork and", "chicken and", "sushi",
                    "omelet", "scrambled egg", "hash", "pilaf", "rice bowl", "kabob", "kebab", "gyro",
                    "fried chicken", "nuggets", "fish sticks", "general tso", "kung pao", "orange chicken",
                    "sweet and sour", "teriyaki chicken", "puerto rican style", "empanada", "pupusa"), desc)
             & ~INFANT & ~DESSERT & ~BEVERAGE
             & ~has(kw("sauce,", "sauce$", "dressing", "gravy", "seasoning", "dip\\b", "spread"), desc)
             & ~has(kw("as ingredient", "unprepared", "dry mix", "raw\\b", "wrappers?", "hot dog bun",
                       "rolls?, (?:dinner|hamburger|hot ?dog)", "crackers", "cookies?", "bread\\b(?!.*sandwich)",
                       "noodles, (?:egg|chow mein|japanese|chinese)", "pasta, (?:homemade|fresh|dry|cooked)",
                       "luncheon", "sandwich spread"), desc)
             & ~(desc.str.startswith(("beef,", "pork,", "lamb,", "veal,", "chicken,", "turkey,", "fish,",
                                      "game meat", "crustaceans", "mollusks", "spices", "peppers",
                                      "noodles", "pasta,", "rolls", "bread", "crackers", "cookies",
                                      "cereals", "snacks", "sauce", "cheese,", "rice,"))))
RAW = has(kw("raw\\b", "unprepared", "dry mix", "unheated", "not reconstituted"), desc)

# ---------- 1. authenticity, origin & fusion ----------
REGION_PATTERNS = {
    "italian": kw("italian", "lasagna", "ravioli", "tortellini", "risotto", "gnocchi", "polenta", "pesto",
                  "carbonara", "marinara", "parmigiana", "cannoli", "tiramisu", "biscotti", "focaccia",
                  "prosciutto", "mortadella", "mozzarella", "parmesan", "ricotta", "provolone", "romano",
                  "calzone", "minestrone", "manicotti", "ziti", "fettuccine", "alfredo", "spaghetti",
                  "salami", "capicola", "osso buco", "panettone", "gelato", "espresso", "cappuccino",
                  "bruschetta", "antipasto", "bologna(?!.*sandwich)"),
    "mexican_latin": kw("mexican", "latino", "taco", "burrito", "enchilada", "tamale", "quesadilla", "tostada",
                        "chimichanga", "fajita", "nachos?", "salsa", "guacamole", "refried", "tortilla",
                        "mole\\b", "pozole", "posole", "menudo", "carnitas", "barbacoa", "chorizo", "queso",
                        "horchata", "churro", "pupusa", "empanada", "arepa", "plantain", "puerto rican",
                        "cuban", "dominican", "chile relleno", "chilaquiles", "flan", "arroz", "frijoles",
                        "sofrito", "mofongo", "tres leches", "atole", "jicama", "nopal", "tomatillo",
                        "chipotle", "jalape", "spanish rice", "pico de gallo", "on the border", "taco bell"),
    "chinese": kw("chinese", "chow mein", "lo mein", "egg roll", "spring roll", "wonton", "dim sum", "dumpling",
                  "fried rice", "kung pao", "general tso", "sweet and sour", "moo shu", "hoisin", "bok choy",
                  "szechuan", "sichuan", "chop suey", "orange chicken", "lemon chicken", "egg foo", "bao\\b",
                  "oyster sauce", "black bean sauce", "fortune cookie", "hot and sour", "mandarin"),
    "japanese": kw("japanese", "sushi", "sashimi", "teriyaki", "tempura", "miso", "ramen", "udon", "soba",
                   "edamame", "wasabi", "tofu", "seaweed", "nori", "sake", "mirin", "natto", "matcha",
                   "katsu", "gyoza", "yakitori", "tamari", "dashi", "furikake", "california roll"),
    "korean": kw("korean", "kimchi", "bulgogi", "bibimbap", "gochujang", "japchae"),
    "thai_vietnamese": kw("thai", "pad thai", "pho\\b", "banh mi", "vietnamese", "lemongrass", "fish sauce",
                          "satay", "coconut curry", "spring roll", "tom yum", "larb"),
    "indian": kw("indian", "curry", "masala", "tandoori", "tikka", "naan", "chapati", "roti\\b", "dal\\b", "dhal",
                 "samosa", "biryani", "paneer", "chutney", "ghee", "lassi", "korma", "vindaloo", "pakora",
                 "garam", "raita", "papad", "idli", "dosa", "chana", "turmeric"),
    "middle_eastern_med": kw("greek", "hummus", "falafel", "gyro", "pita", "tahini", "baklava", "tabbouleh",
                             "kabob", "kebab", "shawarma", "feta", "tzatziki", "dolma", "couscous",
                             "baba ghanoush", "spanakopita", "moussaka", "halvah", "halva", "lebanese",
                             "turkish", "persian", "za'atar", "labneh", "olives?\\b"),
    "french": kw("french(?! fries| fried| toast| dressing| onion dip)", "croissant", "crepe", "quiche",
                 "souffle", "brioche", "baguette", "bearnaise", "hollandaise", "ratatouille", "bouillabaisse",
                 "brie", "camembert", "roquefort", "gruyere", "fondue", "eclair", "creme brulee", "macarons?\\b",
                 "vichyssoise", "pate\\b", "foie gras", "coq au vin", "bechamel", "cassoulet", "au gratin"),
    "german_eastern_eu": kw("german", "sauerkraut", "bratwurst", "knockwurst", "strudel", "pierogi", "kielbasa",
                            "pretzel", "schnitzel", "spaetzle", "goulash", "borscht", "blintz", "latke",
                            "knish", "polish", "hungarian", "pumpernickel", "rye bread", "limburger", "muenster"),
    "southern_us_cajun": kw("cajun", "creole", "gumbo", "jambalaya", "grits", "hush pupp", "collards",
                            "cornbread", "biscuits? and gravy", "chitterling", "hoppin", "fried okra",
                            "po'? ?boy", "etouffee", "pimento cheese", "black-eyed", "hominy", "southern",
                            "fried green tomato", "pecan pie", "sweet potato pie", "chicken fried steak",
                            "soul food"),
    "native_american_alaskan": kw("alaska native", "navajo", "hopi", "apache", "shoshone", "native american",
                                  "fry bread", "frybread", "pemmican", "northern plains indians",
                                  "southwest", "acorn", "blue corn", "mutton", "seal\\b", "whale", "caribou",
                                  "muktuk", "agutuk", "ookpeeyuk"),
    "caribbean_hawaiian": kw("jamaican", "jerk", "hawaiian", "poi\\b", "spam musubi", "taro", "breadfruit",
                             "ackee", "callaloo", "cassava", "yuca", "lomi", "kalua", "haupia", "poke\\b"),
}
region_hits = {r: has(p) for r, p in REGION_PATTERNS.items()}
region_df = pd.DataFrame(region_hits)
region_count = region_df.sum(axis=1)

# Well-known fusion dishes (American-Chinese, Tex-Mex, cross-cuisine mash-ups).
FUSION_DISHES = kw("general tso", "orange chicken", "chop suey", "fortune cookie", "crab rangoon",
                   "egg foo", "sweet and sour", "lemon chicken", "california roll", "sushi roll, california",
                   "nachos", "fajita", "chimichanga", "chili con carne", "taco salad", "tex-?mex",
                   "taco bell", "on the border", "doritos", "taco-flavor", "nacho-flavor", "burrito supreme",
                   "hawaiian pizza", "bbq chicken pizza", "buffalo chicken (?:pizza|wrap)", "teriyaki (?:burger|pizza)",
                   "breakfast burrito", "rice bowl", "spam musubi", "korean taco", "sushi burrito",
                   "curry (?:ketchup|wurst)", "currywurst", "pizza bagel", "bagel pizza", "french bread pizza",
                   "cuban sandwich", "philly", "kimchi fried rice", "poke bowl",
                   "thai coconut curry", "jalapeno popper", "cheesesteak")
# Signs of a westernised / convenience take rather than a traditional preparation.
WESTERNIZED = kw("frozen entree", "diet frozen meal", "frozen meal", "microwave", "lean cuisine", "healthy choice",
                 "weight watchers", "stouffer", "banquet", "reduced fat", "low fat", "fat free", "reduced sodium",
                 "light\\b", "lite\\b", "canned, condensed", "school", "fast foods?", "chips", "flavor",
                 "kraft", "hormel", "campbell", "mcdonald", "burger king", "wendy", "pizza hut", "domino",
                 "papa john", "kfc", "popeyes", "applebee", "t\\.g\\.i", "denny", "olive garden", "cracker barrel",
                 "with american cheese", "cheese sauce", "velveeta", "chinese restaurant", "from frozen",
                 "frozen, (?:un)?prepared", "smart soup")
fusion = has(FUSION_DISHES) | ((region_count >= 2) & MAIN_DISH)
fusion &= ~INFANT

# Authentic = clear single regional identity, a traditional named dish/ingredient, not a westernised
# convenience product and not a fusion dish.
TRADITIONAL = kw("alaska native", "navajo", "hopi", "apache", "shoshone", "northern plains", "puerto rican style",
                 "restaurant, latino", "pupusa", "tamale", "pozole", "posole", "menudo", "carnitas", "barbacoa",
                 "mole\\b", "mofongo", "arepa", "empanada", "sushi roll, (?:eel|salmon|tuna)", "sushi roll tuna",
                 "sashimi", "miso", "natto", "kimchi", "bibimbap", "bulgogi", "pad thai", "pho\\b", "banh mi",
                 "biryani", "dal\\b", "dhal", "samosa", "naan", "chapati", "paneer", "tikka", "tandoori",
                 "hummus", "falafel", "tabbouleh", "baba ghanoush", "spanakopita", "moussaka", "dolma",
                 "risotto", "gnocchi", "osso buco", "polenta", "cannoli", "tiramisu", "prosciutto", "pesto",
                 "carbonara", "bouillabaisse", "ratatouille", "cassoulet", "coq au vin", "croissant",
                 "brioche", "baguette", "gumbo", "jambalaya", "etouffee", "grits", "hush pupp", "collards",
                 "pierogi", "borscht", "sauerkraut", "bratwurst", "knish", "latke", "blintz", "fry ?bread",
                 "poi\\b", "kalua", "lomi", "haupia", "tortilla, corn", "refried beans", "queso (?:cotija|anejo|fresco|chihuahua|asadero|blanco)",
                 "stew, .*mutton", "tripe soup", "pho", "dumpling", "wonton", "dim sum", "baklava", "halva",
                 "feta", "roquefort", "brie", "camembert", "gruyere", "parmesan", "mozzarella", "ricotta",
                 "chorizo", "horchata", "atole", "tomatillo", "nopal", "plantain", "cassava", "taro",
                 "seaweed", "tofu", "tempura", "udon", "soba", "ramen(?!.*noodle soup, .*dry)", "edamame",
                 "curry", "lassi", "ghee", "tzatziki", "couscous", "pita", "gyro", "kabob", "kebab")
authentic_regional = (region_count >= 1) & has(TRADITIONAL) & ~fusion & ~has(WESTERNIZED) & ~INFANT \
    & ~(RAW & ~has(kw("alaska native", "navajo", "hopi", "apache", "shoshone", "northern plains"), desc))
cuisine_region = region_df.apply(lambda r: "|".join(c for c, v in r.items() if v), axis=1)

STREET = kw("taco", "burrito", "hot dog", "frankfurter", "corn dog", "pretzels?, soft", "soft pretzel",
            "kabob", "kebab", "gyro", "shawarma", "falafel", "empanada", "pupusa", "tamale", "arepa",
            "egg roll", "spring roll", "dumpling", "bao\\b", "banh mi", "churro", "funnel cake", "elote",
            "nachos", "quesadilla", "sliders?", "cheeseburger", "hamburger(?! helper)", "pad thai", "french fries", "fries",
            "fried chicken", "chicken wings?", "chicken nuggets", "chicken tenders", "fish and chips",
            "pizza, .*slice", "pizza", "sandwich", "wrap\\b", "sub\\b", "hoagie", "po'? ?boy", "cheesesteak",
            "fast foods", "cotton candy", "kettle corn", "caramel corn", "popcorn", "roasted peanuts",
            "doughnut", "donut", "fry ?bread", "pierogi", "knish", "samosa", "satay", "spam musubi",
            "shaved ice", "snow cone", "sno cone", "fresh-?squeezed lemonade", "roasted chestnut",
            "corn on the cob", "street", "onion rings", "fish sticks", "chimichanga", "tostada")
street_food = has(STREET, desc) & ~INFANT & ~RAW & ~has(kw("frozen, unprepared", "as ingredient", "dry mix", "school", "cookies?",
    "crackers", "smoothie", "rolls?, .*(?:bun|hot ?dog)", "bun\\b", "sandwich spread"), desc)

# ---------- 2. occasion, meal course & context ----------
COMFORT = kw("macaroni and cheese", "macaroni or noodles with cheese", "mac.{0,5}cheese", "mashed potato", "pot pie", "meatloaf", "meat loaf",
             "chicken noodle soup", "soup, chicken noodle", "grilled cheese", "lasagna", "casserole", "chili\\b",
             "stew", "pot roast", "fried chicken", "chicken, .*fried", "biscuits?\\b", "gravy", "dumpling",
             "pancake", "waffle", "french toast", "cinnamon roll", "sweet roll", "brownie", "cookie",
             "chocolate chip", "apple pie", "pie\\b", "cobbler", "bread pudding", "rice pudding", "pudding",
             "ice cream", "hot chocolate", "cocoa", "mashed", "au gratin", "scalloped potato", "potatoes, hashed",
             "hash brown", "tater tots", "shepherd", "cornbread", "spaghetti", "meatballs", "ravioli",
             "beef and noodles", "chicken and dumplings", "chicken and rice", "cream of", "tomato soup",
             "soup, tomato", "pizza", "cheeseburger", "hamburger", "hot dog", "french fries", "donut",
             "doughnut", "sloppy joe", "enchilada", "tamale", "ramen", "congee", "grits", "stroganoff",
             "goulash", "fettuccine alfredo", "alfredo", "cheesecake", "cake\\b", "cupcake", "s'mores",
             "mac and cheese", "cheese sauce", "queso dip", "fondue", "risotto", "polenta", "pierogi", "gumbo",
             "chowder", "bisque", "biscuits? and gravy", "chicken fried steak", "fish and chips", "baked beans",
             "beans, baked", "creamed", "stuffing", "dressing, bread", "noodle soup")
comfort_food = has(COMFORT, desc) & ~INFANT & ~has(kw("diet", "reduced fat", "low fat", "fat free", "light\\b",
                                                      "sugar free", "unprepared", "dry mix", "lite\\b", "sauce,", "sauce$",
                                                      "enchilada sauce", "cereals", "gluten-free", "rolls?,", "bun\\b",
                                                      "pie crust", "dry form", "flakes", "as ingredient"), desc) & ~RAW \
    & ~desc.str.startswith(("beef,", "pork,", "lamb,", "veal,"))
# rich flavour profile support: satisfying, energy dense prepared foods
comfort_food |= (MAIN_DISH & (n["energy_kcal"] >= 200) & ((n["fat_g"] >= 10) | (n["carb_g"] >= 25))
                 & has(kw("cheese", "cream", "butter", "gravy", "fried", "noodles", "potato", "bread"), desc))

# Weeknight dinner: a savoury main that is quick / low-effort (ready meals, skillet, stir-fry, pasta, tacos)
QUICK_MAIN = kw("stir.?fr", "fried rice", "tacos?", "burrito", "quesadilla", "fajita", "spaghetti", "pasta",
                "macaroni", "noodles", "lo mein", "chow mein", "pad thai", "rice bowl", "omelet", "scrambled",
                "frittata", "chili\\b(?! pepper|, green|, red|s)", "soup", "curry", "burger", "sloppy joe", "meatballs", "sandwich",
                "wrap\\b", "pizza", "frozen entree", "frozen meal", "diet frozen meal", "entree", "dinner",
                "skillet", "hamburger helper", "teriyaki", "sweet and sour", "kung pao", "chicken and vegetables",
                "beef and vegetables", "with rice", "with noodles", "with vegetables", "with pasta",
                "grilled chicken", "chicken breast, (?:baked|broiled|grilled|sauteed|roasted)",
                "fish, .*(?:baked|broiled)", "salmon, .*(?:cooked|baked)", "pork chop", "sausage", "fish sticks",
                "chicken nuggets", "chicken tenders", "enchilada", "jambalaya", "gumbo", "salad with")
SLOW_EFFORT = kw("pot roast", "braised", "roast, ", "whole", "turkey, whole", "prime rib", "smoked", "barbecue",
                 "brisket", "stew(?!ed)", "lasagna(?! .*frozen)", "casserole", "souffle", "wellington",
                 "restaurant", "carcass", "unprepared", "dry mix")
weeknight_dinner = MAIN_DISH & has(QUICK_MAIN, desc) & ~has(SLOW_EFFORT, desc) \
    & ~has(kw("breakfast", "biscuit", "egg sandwich", "pancake", "waffle", "school", "snack", "soup, .*dry",
              "bouillon", "broth"), desc) \
    & ~(desc.str.contains("frozen entree|frozen meal|lasagna") & False)
weeknight_dinner |= MAIN_DISH & has(kw("frozen entree", "frozen meal", "diet frozen meal", "lasagna.*frozen",
                                       "frozen, prepared"), desc)

HOLIDAY = kw("thanksgiving", "christmas", "holiday", "easter", "hanukkah", "passover", "matzo", "matzah", "latke",
             "turkey, whole", "turkey, .*roasted", "turkey, .*(?:breast|dark meat|light meat).*roast",
             "roasted turkey", "stuffing", "dressing, bread", "cranberry sauce", "cranberr(?:y|ies), (?:dried|raw)", "cranberry relish", "pumpkin pie",
             "pecan pie", "sweet potato, casserole", "sweet potato pie", "candied yams", "yams, candied",
             "green bean casserole", "gravy, turkey", "turkey gravy", "eggnog", "egg nog", "fruitcake",
             "fruit cake", "gingerbread", "candy cane", "peppermint", "panettone", "stollen", "ham, .*(?:whole|glazed|honey|spiral|roasted)",
             "honey ham", "glazed ham", "prime rib", "rib roast", "leg of lamb", "lamb, .*leg, whole",
             "hot cross bun", "king cake", "mooncake", "wedding cake", "birthday cake", 
             "champagne", "sparkling wine", "deviled egg", "cheese ball", "cheesecake", "mincemeat",
             "chestnuts, .*roasted", "wassail", "cider, hot", "tamale", "pozole", "posole", "bread pudding",
             "challah", "brisket", "kugel", "haroset", "rugelach", "hamantaschen", "pie, (?:apple|pumpkin|pecan|cherry|mince)")
holiday_celebration = has(HOLIDAY, desc) & ~INFANT & ~RAW \
    & ~has(kw("beans, cranberry", "egg, turkey", "cranberry(?:-\\w+)? juice drink", "juice cocktail",
              "chestnuts, (?:chinese|japanese|european), (?:dried|boiled|steamed)", "oyster"), desc)

GAME_DAY = kw("chicken wings?", "chicken, .*wings?,", "buffalo", "nachos", "tortilla chips", "potato chips", "chips",
              "dip\\b", "salsa", "guacamole", "queso", "pizza", "pizza rolls", "sliders?",
              "hot dog", "frankfurter", "bratwurst", "kielbasa", "pigs in a blanket", "popcorn", "pretzels",
              "snack mix", "chex mix", "party mix", "jalapeno popper", "poppers", "mozzarella sticks",
              "cheese sticks", "onion rings", "potato skins", "french fries", "fries", "chili\\b(?! pepper|, green|, red|s)",
              "beer", "ale\\b", "lager", "cocktail meatballs", "cocktail sausage", "deviled egg", "spinach dip",
              "bean dip", "cheese puffs", "cheetos", "doritos", "cheese curls", "corn chips", "fritos",
              "pork rinds", "chicharron", "jerky", "peanuts", "mixed nuts", "quesadilla", "taco", "egg roll",
              "spring roll", "chicken nuggets", "chicken tenders", "chicken strips", "barbecue", "bbq",
              "ribs", "pulled pork", "cheese crackers", "crackers, cheese", "cheese ball")
game_day_snack = has(GAME_DAY, desc) & ~INFANT & ~has(kw("baked, fat free", "unsalted", "low sodium", "diet",
    "carbonated", "root beer", "ginger ale", "caramel dip", "as ingredient"), desc) \
    & ~desc.str.startswith(("beef,", "pork,", "spices", "seeds", "oil", "peppers"))

KID = kw("macaroni and cheese", "mac.{0,5}cheese", "chicken nuggets", "chicken tenders", "chicken strips",
         "fish sticks", "grilled cheese", "peanut butter", "jelly", "jam\\b", "hot dog", "frankfurter",
         "corn dog", "pizza", "spaghetti", "meatballs", "pancake", "waffle", "french toast", "cereal",
         "cereals ready-to-eat", "yogurt", "applesauce", "apple juice", "grape juice", "fruit cocktail",
         "fruit snacks", "fruit leather", "fruit roll", "graham", "animal crackers", "goldfish", "cheese sticks",
         "string cheese", "cheese, american", "cheese, cheddar", "cookie", "brownie", "cupcake", "muffin",
         "popsicle", "ice cream", "pudding", "gelatin dessert", "jell-?o", "chocolate milk", "milk, chocolate",
         "milk, .*flavored", "banana", "apples?,", "grapes", "strawberr", "carrots?, baby", "corn, sweet",
         "peas, green", "french fries", "tater tots", "potato, .*mashed", "mashed potato", "rice krispies",
         "cheerios", "pop-tarts?", "toaster pastr", "uncrustables", "sandwich, peanut butter", "noodles",
         "ramen", "quesadilla", "taco", "chicken drumstick", "chicken, .*drumstick", "hamburger", "cheeseburger",
         "slider", "pretzels", "popcorn", "crackers", "raisins", "school", "kids?\\b", "child", "lunchables",
         "juice drink", "fruit punch", "smoothie", "babyfood", "toddler", "cheese pizza", "bagel", "biscuit")
KID_AVOID = kw("spicy", "hot\\b", "jalape", "chipotle", "habanero", "cayenne", "buffalo", "wasabi", "curry",
               "horseradish", "anchov", "sardine", "liver", "kidney", "tongue", "tripe", "heart", "brain",
               "gizzard", "chitterling", "blood", "caviar", "roe\\b", "oyster", "clam", "mussel", "squid",
               "octopus", "eel\\b", "coffee", "espresso", "tea\\b", "energy drink", "caffeine", "blue cheese",
               "roquefort", "limburger", "gorgonzola", "kimchi", "sauerkraut", "olive", "brussels", "bitter",
               "diet", "sugar free", "artificial sweetener", "protein powder", "supplement", "formulated bar",
               "slim-?fast", "infant formula", "tofu", "natto", "sushi", "raw\\b.*(?:beef|pork|fish|chicken|egg)",
               "alcohol", "seal\\b", "whale", "caribou", "mutton", "game meat", "venison")
kid_friendly = has(KID, desc) & ~has(KID_AVOID, desc) & ~ALCOHOL & ~(RAW & ~has(kw("fruit", "apples?", "banana", "grapes", "strawberr", "carrots?, baby"), desc)) \
    & ~desc.str.startswith(("infant formula",))

# ---------- 3. flavour profiles ----------
HOT_STRONG = kw("habanero", "ghost pepper", "scotch bonnet", "thai chili", "cayenne", "hot sauce", "sriracha",
                "sauce, hot", "hot chili", "chili powder", "chili pepper", "peppers, hot", "red pepper flakes",
                "crushed red pepper", "chipotle", "jalape", "serrano", "buffalo", "jerk", "vindaloo", "kimchi",
                "gochujang", "szechuan", "sichuan", "kung pao", "general tso", "hot and spicy", "spicy",
                "extra hot", "flamin", "hot wings", "wasabi", "horseradish", "harissa", "tabasco",
                "pepper, red or cayenne", "curry powder", "curry paste", "salsa, (?:hot|picante)", "picante",
                "hot pepper", "pepperoncini", "chile relleno", "with green chil", "green chil", "red chil",
                "hot italian sausage", "pepper jack", "andouille", "cajun", "tandoori", "tikka masala",
                "mole\\b", "chorizo", "enchilada sauce", "taco sauce", "chili\\b", "chili con",
                "curry", "sweet and spicy", "hot (?:sauce|pepper|chil|wings|salsa|italian|mustard|and spicy|links?|sausage)")
spicy = has(HOT_STRONG) & ~INFANT & ~has(kw("mild", "not hot", "no heat", "chili, ?.*without beans.*mild",
                                            "bell pepper", "sweet pepper", "game meat, buffalo",
                                            "buffalo, water"), desc)
heat_level = np.select(
    [has(kw("habanero", "ghost pepper", "scotch bonnet", "extra hot", "flamin", "vindaloo", "thai chili",
            "sauce, hot", "hot sauce", "peppers, hot", "hot chili", "cayenne", "sriracha", "tabasco", "buffalo"))
     & spicy,
     spicy],
    ["hot", "medium"], default="none")

UMAMI = kw("soy sauce", "tamari", "miso", "fish sauce", "oyster sauce", "worcestershire", "anchov", "sardine",
           "bonito", "dashi", "seaweed", "kelp", "nori", "kombu", "mushroom", "shiitake", "porcini", "truffle",
           "parmesan", "romano", "pecorino", "gruyere", "aged cheddar", "cheese, cheddar", "roquefort",
           "blue cheese", "gouda", "swiss", "asiago", "tomato paste", "tomato sauce", "sun-dried tomato",
           "marinara", "spaghetti sauce", "pizza sauce", "ketchup", "catsup", "bouillon", "broth", "stock\\b",
           "consomme", "gravy", "bacon", "ham\\b", "prosciutto", "salami", "pepperoni", "jerky", "cured",
           "smoked", "sausage", "cured meat", "beef, .*(?:roasted|broiled|grilled)", "steak", "pot roast",
           "roast beef", "meat extract", "yeast extract", "marmite", "vegemite", "msg", "monosodium",
           "nutritional yeast", "teriyaki", "hoisin", "black bean sauce", "kimchi", "natto", "tempeh",
           "fermented", "shellfish", "shrimp", "crab", "lobster", "scallop", "clam", "oyster", "mussel",
           "tuna", "mackerel", "salmon", "caviar", "roe\\b", "cheeseburger", "pizza", "ramen", "pho",
           "french onion", "onion soup", "beef stew", "chili con carne", "bolognese", "meat sauce", "with meat",
           "clam chowder", "caramelized onion", "kung pao", "general tso", "pad thai", "fried rice",
           "lo mein", "chow mein", "chop suey", "egg foo", "stir.?fr", "walnut", "green tea", "sauerkraut", "olive tapenade")
INHERENT_UMAMI = kw("mushroom", "shiitake", "seaweed", "kelp", "nori", "kombu", "tomato", "cheese", "anchov",
                    "sardine", "miso", "soy sauce", "fish sauce", "oyster sauce", "worcestershire", "cured",
                    "smoked", "bacon", "ham\\b", "prosciutto", "salami", "pepperoni", "jerky", "yeast extract",
                    "bouillon", "broth", "kimchi", "natto", "tempeh", "fermented", "caviar", "roe\\b")
umami_rich = (has(UMAMI) & ~INFANT & ~DESSERT & ~BEVERAGE & (~RAW | has(INHERENT_UMAMI, desc))
              & ~has(kw("french fr", "steak cut", "steak fries"), desc)) \
    | (~INFANT & ~DESSERT & ~BEVERAGE & (n["protein_g"] >= 15) & (n["sodium_mg"] >= 500)
       & has(kw("cooked", "roasted", "grilled", "broiled", "braised", "smoked", "cured", "fried", "stewed",
                "with", "sauce", "canned")))

TANGY = kw("lemon", "lime", "citrus", "orange", "grapefruit", "tangerine", "clementine", "yuzu", "kumquat",
           "vinegar", "vinaigrette", "pickle", "pickled", "sauerkraut", "kimchi", "relish", "capers?\\b",
           "mustard", "buttermilk", "sour cream", "cream, sour", "yogurt, plain", "yogurt, greek, plain",
           "kefir", "cranberr", "tamarind", "sumac", "rhubarb", "gooseberr", "currant", "sour cherr",
           "cherries, sour", "passion fruit", "passion-fruit", "pomegranate", "kiwi", "pineapple",
           "sweet and sour", "sweet & sour", "barbecue", "bbq", "ketchup", "catsup", "salsa", "pico de gallo",
           "ceviche", "chutney", "tomatillo", "pad thai", "(?<!includes )sourdough", "sour\\b", "tart\\b", "tangy", "zesty", "zest",
           "italian dressing", "salad dressing, italian", "french dressing", "salad dressing, french",
           "thousand island", "coleslaw", "cole slaw", "tzatziki", "labneh", "lemonade", "limeade",
           "margarita", "lemon chicken", "orange chicken", "key lime", "green mango", "sorrel",
           "hot and sour", "tomatoes?, green", "vinegar-based", "hollandaise")
tangy_zesty = has(TANGY) & ~INFANT & ~has(kw("biscuits?", "tart, breakfast", "toaster pastr", "heavy syrup", "extra heavy syrup", "fruit salad", "fruit cocktail"), desc)
# low-sugar yoghurts / citrus-heavy items without masking sugar count, heavily sweetened ones do not
tangy_zesty &= ~(DESSERT & (n["sugars_g"] >= 35) & ~has(kw("lemon", "lime", "key lime", "cranberr", "rhubarb")))

# ---------- 4. technical complexity ----------
ADVANCED = kw("souffle", "wellington", "foie gras", "pate\\b", "terrine", "confit", "bouillabaisse", "consomme",
              "hollandaise", "bearnaise", "beurre blanc", "bechamel", "veloute", "demi-?glace", "risotto",
              "croissant", "puff pastry", "phyllo", "filo", "baklava", "eclair", "cream puff", "profiterole",
              "macarons?\\b", "creme brulee", "tiramisu", "mille-?feuille", "napoleon", "cannoli", "strudel",
              "brioche", "danish pastry", "(?<!includes )sourdough", "sushi", "sashimi", "caviar", "lobster", "truffle",
              "duck", "goose", "quail", "pheasant", "squab", "osso buco", "sweetbread",
              "bone marrow", "escargot", "snail", "octopus", "squid", "tempura", "dim sum", "dumpling",
              "wonton", "gyoza", "pierogi", "gnocchi", "ravioli, (?:homemade|fresh)", "tamale", "mole\\b",
              "coq au vin", "cassoulet", "beef bourguignon", "paella", "biryani", "moussaka", "spanakopita",
              "cheesecake", "layer cake", "angel food", "sponge cake", "chiffon", "meringue", "custard",
              "flan\\b", "creme caramel", "pavlova", "baked alaska", "frangipane", "praline", "caramel",
              "toffee", "fudge", "gelato", "sorbet", "mousse", "ganache", "tart\\b", "tarte", "quiche",
              "crepe", "blintz", "timbale", "galantine", "roulade", "rack of lamb", "prime rib",
              "beef, .*tenderloin", "filet mignon", "chateaubriand", "scallops?\\b(?!.*imitation)", "abalone", "sea urchin",
              "roe\\b(?!.*imitation)", "fondue", "consomm", "mofongo", "empanada", "samosa", "naan", "croissant",
              "puff", "choux", "laminated", "tempering", "sous vide", "flambe", "caramelized")
advanced_gourmet = has(ADVANCED, desc) & ~INFANT & ~has(WESTERNIZED, desc) \
    & ~has(kw("unprepared", "dry mix", "frozen, (?:unprepared|prepared)", "canned", "school", "snacks?",
              "candies", "commercially prepared", "ready-to-eat", "sweetened, with added", "fast foods",
              "restaurant, family"), desc) \
    & ~desc.str.startswith(("snacks", "candies", "cereals")) \
    & ~(RAW & ~has(kw("sushi", "sashimi", "caviar", "sea urchin"), desc)) \
    & ~has(kw("wrappers?", "macaroni", "pasta salad", "corn flour patty", "egg,", "goose egg", "duck sauce",
              "candy", "surimi", "imitation"), desc)

ONE_POT = kw("soup", "macaroni or noodles with cheese", "stew", "chili\\b", "chili con carne", "chowder", "bisque", "gumbo", "jambalaya", "paella",
             "risotto", "pilaf", "goulash", "curry", "casserole", "pot roast", "hot pot", "braise", "stir.?fr",
             "fried rice", "lo mein", "chow mein", "pad thai", "skillet", "one-?pot", "one-?pan", "sheet pan",
             "frittata", "omelet", "scrambled egg", "egg omelet", "hash\\b", "hash, ", "corned beef hash",
             "shakshuka", "congee", "porridge", "oatmeal", "grits", "polenta", "dal\\b", "dhal", "pozole",
             "posole", "menudo", "pho", "ramen", "noodle soup", "chicken and rice", "arroz con pollo",
             "rice and beans", "beans and rice", "red beans and rice", "hoppin", "beef and noodles",
             "stroganoff", "sloppy joe", "chili mac", "hamburger helper", "macaroni and cheese",
             "mac and cheese", "stewed", "with gravy", "etouffee", "tagine", "cassoulet", "boiled dinner",
             "pot pie", "fajita", "sauteed", "stir fried", "picadillo", "ropa vieja", "puerto rican style",
             "beef and vegetables", "chicken and vegetables", "pork and vegetables", "baked beans",
             "beans, baked", "refried beans", "succotash", "ratatouille", "mixed vegetables")
one_pot_one_pan = has(ONE_POT, desc) & ~INFANT & ~DESSERT & ~BEVERAGE \
    & ~has(kw("dry mix", "dry,", "condensed", "unprepared", "stewed, (?:with|without) added sugar",
              "dried.*stewed", "cereal", "oatmeal cookie", "soup mix", "bouillon", "broth", "stock\\b",
              "consomme", "sauteed, skin", "sushi", "bread", "french fries", "fries", "chow mein noodles",
              "noodles, chow mein", "as ingredient", "stewing", "granola", "bars?\\b", "dip\\b"), desc) & ~RAW \
    & ~desc.str.startswith(("cereals", "beef,", "pork,", "lamb,", "veal,", "peppers", "snacks"))
# a dish that needs separate components (e.g. breaded & fried, layered, stuffed, with sides) is not one-pot
one_pot_one_pan &= ~has(kw("stuffed", "layered", "breaded", "with (?:mashed|french fries|side)", "lasagna",
                           "wellington", "casserole with bread", "crust", "sandwich", "on (?:bun|roll|bread)"), desc)

# ---------- assemble ----------
LABEL_COLS = ["authentic_regional", "fusion_cuisine", "street_food_style", "comfort_food", "weeknight_dinner",
              "holiday_celebration", "game_day_snack", "kid_friendly", "spicy", "umami_rich", "tangy_zesty",
              "advanced_gourmet", "one_pot_one_pan"]
NUTRIENT_COLS = ["protein_g", "fat_g", "carb_g", "energy_kcal", "fiber_g", "sodium_mg", "cholesterol_mg",
                 "satfat_g", "sugars_g"]
recipe_classes = price_mapped_nutrients_df[["fdc_id", "description", "product", "adj", "verb"] + NUTRIENT_COLS].copy()
recipe_classes = recipe_classes.assign(**{
    "authentic_regional": authentic_regional,
    "fusion_cuisine": fusion,
    "street_food_style": street_food,
    "comfort_food": comfort_food,
    "weeknight_dinner": weeknight_dinner,
    "holiday_celebration": holiday_celebration,
    "game_day_snack": game_day_snack,
    "kid_friendly": kid_friendly,
    "spicy": spicy,
    "umami_rich": umami_rich,
    "tangy_zesty": tangy_zesty,
    "advanced_gourmet": advanced_gourmet,
    "one_pot_one_pan": one_pot_one_pan,
})
recipe_classes[LABEL_COLS] = recipe_classes[LABEL_COLS].astype(int)
# rows with no specific class are "generic"; generic is exclusive and never co-occurs with another class
recipe_classes["generic"] = (recipe_classes[LABEL_COLS].sum(axis=1) == 0).astype(int)
is_generic = recipe_classes["generic"] == 1
recipe_classes["heat_level"] = heat_level
recipe_classes["cuisine_region"] = cuisine_region.where(~is_generic, "")
ALL_LABEL_COLS = LABEL_COLS + ["generic"]
recipe_classes["labels"] = recipe_classes[ALL_LABEL_COLS].apply(
    lambda r: "|".join(c for c, v in r.items() if v), axis=1)
recipe_classes["n_labels"] = recipe_classes[ALL_LABEL_COLS].sum(axis=1)
assert (recipe_classes.loc[is_generic, LABEL_COLS].sum(axis=1) == 0).all()
assert (recipe_classes["n_labels"] >= 1).all()

recipe_classes.to_csv("./input_stage2/recipe_classes.csv", index=False)
print(recipe_classes[ALL_LABEL_COLS].sum().sort_values(ascending=False))
print("generic share:", recipe_classes["generic"].mean().round(3),
      "| mean labels/non-generic row:", recipe_classes.loc[~is_generic, "n_labels"].mean().round(2))